# Policy Gradient Theorem (Sutton et al., 1999)

## REINFORCE Works. What's Left to Study? 😂

REINFORCE (1992) already proved that we can optimize a policy directly using sampled rewards, without knowing the environment's dynamics.

So what's still missing?

### Remaining Pain Points

1. High Variance — Noisy rewards make learning inefficient.

2. Monte Carlo Dependence — Can we use learned value functions instead of sampled returns?

3. Convergence — Does an unbiased gradient estimator guarantee convergence?

## What Does the Policy Gradient Theorem Solve?

1 is key motivation. 

2 is the key breakthrough.

---

# Sutton et al. (1999) — Core Idea

**Problem**

REINFORCE can directly optimize a parameterized policy using Monte Carlo returns, but its gradient estimates have high variance, making learning slow.

**Core Idea**

Can we use a learned value function to help estimate the policy gradient, instead of relying directly on sampled Monte Carlo returns?

More precisely: can we approximate action values or advantages while still obtaining the **correct policy gradient**?

**Abstraction Level**

- Whether to use Q or Advantage: important to the theory, but we don't need their implementation details yet.
- How to train the value estimator (MC, TD, etc.): not the main focus here.
- The deeper question: **Under what conditions does value function approximation + policy gradient optimization work mathematically?**

**Main Contribution**

They established conditions under which approximate value functions can produce the exact policy gradient, and proved convergence guarantees for a specific form of policy iteration.

# Theorem 1 — Policy Gradient Theorem

## 1. Intuition — The Amusement Park

Imagine managing an amusement park.

At the entrance, visitors can go left or right. Your goal is to adjust the probability of each choice to maximize overall visitor happiness.

**REINFORCE's approach:**

Let visitors explore the park, record their experiences, and use their actual happiness scores to update the policy.

But even visitors who choose the same direction may have very different experiences:

- Visitor A goes left and gets +10 happiness.
- Visitor B goes left but encounters bad weather and gets +2.
- Visitor C goes left and gets +8.

These individual experiences are noisy.

**What if, instead, we knew the expected happiness of going left?**

Rather than relying on one visitor's experience, we could use the average outcome across all possible future experiences.

That's the intuition behind the action-value function:

$$
Q^\pi(s,a)
$$

It represents the expected long-term return from taking action $a$ in state $s$, then following policy $\pi$.

## 2. Key Insight — 30,000-Foot View

REINFORCE uses a sampled return:

$$
G_t
$$

But the true action-value function is the expected return:

$$
Q^\pi(s_t,a_t)=\mathbb{E}[G_t\mid s_t,a_t]
$$

This is the episodic return formulation; in the average-reward setting, returns are adjusted relative to the average reward.

The distinction:

- **Monte Carlo return:** What actually happened in one experience?
- **Action value:** What happens on average across possible future experiences?

Using an expected value removes some of the randomness in individual returns.

But there is a mathematical challenge.

Changing the policy changes not only action probabilities but also which states we visit in the future.

The state visitation distribution $d^\pi(s)$ itself depends on the policy parameters $\theta$.

Can we still express the **exact gradient of long-term reward** using $Q^\pi$, without explicitly calculating how the **entire state distribution changes**?

This is the role of the Policy Gradient Theorem.

## 3. Conclusion — Policy Gradient Theorem

Sutton et al. present the following result:

$$
\boxed{
\nabla_\theta \rho
=
\sum_s d^\pi(s)
\sum_a
\nabla_\theta \pi_\theta(a\mid s)
Q^\pi(s,a)
}
$$

Where:

- $\rho$: Long-term average reward.
- $d^\pi(s)$: Stationary state visitation distribution.
- $\pi_\theta(a\mid s)$: Probability of choosing action $a$ in state $s$.
- $Q^\pi(s,a)$: Long-term action value, defined relative to average reward in this formulation.

**The remarkable result:**

We can express the exact policy gradient using the true action-value function, **without explicitly computing**:

$$
\nabla_\theta d^\pi(s)
$$

The state distribution still appears in the formula as a weighting factor, but we **don't need its derivative**! 

Sampling states from the current policy is relatively straightforward. But computing its derivative is much harder.

This makes the gradient suitable for estimation from experience.

**Next question (Theorem 2):** In practice, we don't know the true $$Q^\pi$$. Can we replace it with a learned approximation and still obtain the correct policy gradient?

# Theorem 2 — Policy Gradient with Function Approximation

## 1. Problem — What if We Don't Know the True Q?

Theorem 1 showed that we can express the exact policy gradient using the true action-value function:

$$
\nabla_\theta \rho
=
\sum_s d^\pi(s)
\sum_a
\nabla_\theta \pi_\theta(a\mid s)Q^\pi(s,a)
$$

But there's a problem:

**In the real world, we don't know the true $$Q^\pi(s,a)$$.**

We can train a function approximator to estimate it:

$$
f_w(s,a)\approx Q^\pi(s,a)
$$

But our estimates will have errors.

If we substitute inaccurate values into the policy gradient formula, won't we get an incorrect gradient?

**Can we use an imperfect value function and still obtain the exact policy gradient?**

## 2. Intuition — The Amusement Park

Imagine we're managing the amusement park again.

We want to decide whether to encourage more visitors to go left or right.

Suppose the true expected happiness is:

| Direction | True Q | Estimated Q |
|---|---:|---:|
| Left | 10 | 9 |
| Right | 6 | 5 |

Our predictions are wrong!

But notice something interesting:

- True difference: 10 − 6 = 4
- Estimated difference: 9 − 5 = 4

**Even though both predictions are wrong, the difference is exactly right.**

For a simple two-action policy, this difference can determine the policy gradient.

So perhaps our value estimator doesn't need to predict every value perfectly.

**It just needs to make the right kinds of errors — errors that don't distort the policy gradient.**

This example illustrates the intuition, not the full mathematical conditions.

## 3. Key Insight — 30,000-Foot View

Sutton et al. discovered that we can impose two conditions on the value function approximator.

**Condition 1 — Learn the value function to a stationary point of its expected squared prediction error.**

The approximation errors must balance in a particular mathematical sense.

**Condition 2 — Compatibility.**

The value function approximator must be structured so that its parameter derivatives match the policy's log-probability derivatives:

$$
\nabla_w f_w(s,a)
=
\nabla_\theta\log\pi_\theta(a\mid s)
$$

Why does this matter?

Together, these conditions ensure that **the value approximation errors are orthogonal to the policy gradient direction**.

In other words, the errors may exist, but their contributions cancel when computing the gradient.

This is the key mathematical insight:

**The value function doesn't need to be perfectly accurate everywhere. It needs to be accurate in the directions that matter for improving the policy.**

## 4. Conclusion — Policy Gradient with Function Approximation

Theorem 2 proves that if the value function approximator satisfies the two conditions above, then:

$$
\boxed{
\nabla_\theta \rho
=
\sum_s d^\pi(s)
\sum_a
\nabla_\theta\pi_\theta(a\mid s)f_w(s,a)
}
$$

Compare this with Theorem 1:

| | Value used | Gradient |
|---|---|---|
| Theorem 1 | True $$Q^\pi$$ | Exact |
| Theorem 2 | Approximate $$f_w$$, satisfying specific conditions | Still exact |

**The remarkable result:**

We don't need a perfect value function to obtain the exact policy gradient!

The approximation can contain errors, as long as those errors cancel in the policy gradient calculation.

This is not a guarantee for arbitrary value estimators. Both the approximation-error condition and compatibility condition are essential.

**Next question (Theorem 3):** Now that we know how to obtain the exact policy gradient using an approximate value function, can we prove that repeatedly updating the policy will converge?




For reward design: e.x, for action "Block", we want to know, the long term expected value:

$$
Q^\pi(s,\mathrm{Block})
=
\mathbb E[G_t\mid s_t=s,a_t=\mathrm{Block}]
$$

we can train a function (Critic) to estimate that:

$$
f_w(s,a)\approx Q^\pi(s,a)
$$

Note:
>we are learning Q function for the current policy, not the optimal Q as in q-learning.

#### Q: If critic estimate good enough, will it be ok?

The real values are:
$$
Q^\pi(s,A)=1.00,\qquad Q^\pi(s,B)=1.01
$$

We estimate:

$$
f_w(s,A)=1.006,\qquad f_w(s,B)=1.004
$$

good enough?

the gradient's direction are opposite!

In [1]:
p = 0.5

def policy_gradient(q):
    return p * (1 - p) * (q[1] - q[0])

true_q = [1.00, 1.01]
estimated_q = [1.006, 1.004]

print(f"True gradient: {policy_gradient(true_q):+.4f}")
print(f"Critic gradient: {policy_gradient(estimated_q):+.4f}")


True gradient: +0.0025
Critic gradient: -0.0005
